In [2]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications import vgg16

# Definindo os caminhos e parâmetros básicos
caminho_treino = './fer2013/train'
caminho_teste = './fer2013/test'
tamanho_lote = 32 # Quantas imagens a IA vê por vez
tamanho_imagem = (48, 48) # Tamanho padrão do FER2013

print("Carregando dados de treinamento...")
dados_treino = tf.keras.utils.image_dataset_from_directory(
    caminho_treino,
    color_mode="rgb", # A VGG16 exige imagens com 3 canais de cor (RGB)
    image_size=tamanho_imagem,
    batch_size=tamanho_lote
)

print("\nCarregando dados de validação (teste)...")
dados_teste = tf.keras.utils.image_dataset_from_directory(
    caminho_teste,
    color_mode="rgb",
    image_size=tamanho_imagem,
    batch_size=tamanho_lote
)

# Aplica o pré-processamento exato que a VGG16 exige
dados_treino = dados_treino.map(lambda x, y: (vgg16.preprocess_input(x), y))
dados_teste = dados_teste.map(lambda x, y: (vgg16.preprocess_input(x), y))

# Carrega a rede pré-treinada sem o "topo" (include_top=False)
base_vgg = VGG16(weights='imagenet', include_top=False, input_shape=(48, 48, 3))

# Congela os pesos para não estragar o que ela já aprendeu
base_vgg.trainable = False

# Construindo o novo "topo" da rede
# Como a base_vgg está com trainable=False, apenas as camadas abaixo aprenderão agora.
modelo_vgg = models.Sequential([
    base_vgg,           # A base da VGG16 (extrator de características)
    layers.Flatten(),   # Transforma os mapas de características 2D em um vetor 1D
    layers.Dense(256, activation='relu'), # Camada densa para processar os padrões extraídos
    layers.Dropout(0.5), # Ajuda a prevenir o overfitting durante o treinamento
    layers.Dense(7, activation='softmax')  # Saída final para as 7 classes de emoções do FER2013
])

# Compilando o modelo
# Usamos o otimizador Adam, que é padrão para esse tipo de tarefa de classificação.
modelo_vgg.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Resumo da Arquitetura
# Observe que a maioria dos parâmetros (da VGG) aparecerá como "Non-trainable params".
modelo_vgg.summary()

# Treinamento de Teste (3 épocas)
# Lembre-se: como as imagens foram carregadas em RGB, o processamento será um pouco mais lento que o baseline.
print("\nIniciando o treinamento com VGG16... Acompanhe a evolução da acurácia.")
historico = modelo_vgg.fit(
    dados_treino,
    validation_data=dados_teste,
    epochs=3
)

Carregando dados de treinamento...
Found 28709 files belonging to 7 classes.

Carregando dados de validação (teste)...
Found 7178 files belonging to 7 classes.
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg16 (Functional)          (None, 1, 1, 512)         14714688  
                                                                 
 flatten (Flatten)           (None, 512)               0         
                                                                 
 dense (Dense)               (None, 256)               131328    
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 7)                 1799      
                                                                 
Total params: 14,847,815
Tra

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50

# 1. Camada de Data Augmentation para ajudar no desbalanceamento
data_augmentation = models.Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.1),
  layers.RandomZoom(0.1),
])

# 2. Base ResNet-50
base_resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(48, 48, 3))
base_resnet.trainable = False

# 3. Modelo Completo
modelo_resnet = models.Sequential([
    data_augmentation, # Aplica o aumento de dados antes de tudo
    base_resnet,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

modelo_resnet.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 4. Treino
print("Iniciando ResNet-50 com Data Augmentation...")
historico_resnet = modelo_resnet.fit(dados_treino, validation_data=dados_teste, epochs=5)